# GroundingDINO Text-Prompted Object Detection

This tutorial demonstrates zero-shot object detection using GroundingDINO
with free-form text prompts.

**Models covered:** `grounding-dino-swin-t`, `grounding-dino-original-swin-t`  
**License:** Apache-2.0 (`grounding-dino-swin-t`) / gated (`original-swin-t`)  
**Task:** Open-vocabulary object detection

In [ ]:
# Cell 2: Setup
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'visionservex', '-q'])

import visionservex as vsx
import numpy as np
print(f'VisionServeX version: {vsx.__version__}')

from visionservex.grounding_dino import list_grounding_dino_models
models = list_grounding_dino_models()
print('Available GroundingDINO models:')
for m in models:
    print(f'  {m["model_id"]:40s} status={m["status"]:15s} license={m["license"]}')

In [ ]:
# Cell 3: Detect with grounding-dino-swin-t (Apache-2.0)
from visionservex.grounding_dino import run_grounding_dino_detect

# Synthetic 640x480 image
image = np.random.randint(0, 255, (480, 640, 3), dtype=np.uint8)

result = run_grounding_dino_detect(
    model_id='grounding-dino-swin-t',
    image=image,
    text_prompt='a cat . a dog . a person .',
    box_threshold=0.35,
    text_threshold=0.25
)

print(f'Model          : grounding-dino-swin-t')
print(f'Status         : {result["status"]}')
print(f'Detections     : {result.get("num_detections", 0)}')
print(f'Latency ms     : {result.get("latency_ms", "N/A")}')
for box in result.get('boxes', []):
    print(f'  [{box["label"]:15s}] conf={box["score"]:.3f} box={box["xyxy"]}')

In [ ]:
# Cell 4: Show bounding boxes
import matplotlib.pyplot as plt
import matplotlib.patches as patches

fig, ax = plt.subplots(figsize=(8, 6))
ax.imshow(image)

colors = plt.cm.Set1.colors
for i, box in enumerate(result.get('boxes', [])):
    x1, y1, x2, y2 = box['xyxy']
    rect = patches.Rectangle((x1, y1), x2-x1, y2-y1,
                               linewidth=2, edgecolor=colors[i % len(colors)], facecolor='none')
    ax.add_patch(rect)
    ax.text(x1, y1-4, f'{box["label"]} {box["score"]:.2f}',
            color=colors[i % len(colors)], fontsize=9, fontweight='bold')

ax.set_title(f'GroundingDINO swin-t  ({result.get("num_detections",0)} detections)')
plt.tight_layout()
plt.savefig('/tmp/gdino_swin_t_result.png', dpi=80)
plt.show()

In [ ]:
# Cell 5: Detect with grounding-dino-original-swin-t
orig_result = run_grounding_dino_detect(
    model_id='grounding-dino-original-swin-t',
    image=image,
    text_prompt='a cat . a dog . a person .',
    box_threshold=0.35,
    text_threshold=0.25
)

print(f'Model          : grounding-dino-original-swin-t')
print(f'Status         : {orig_result["status"]}')
if orig_result['status'] == 'auth_required':
    print(f'Message        : {orig_result.get("message", "HF token required")}')
    print(f'HF repo        : {orig_result.get("hf_repo", "N/A")}')
else:
    print(f'Detections     : {orig_result.get("num_detections", 0)}')
    print(f'Latency ms     : {orig_result.get("latency_ms", "N/A")}')

In [ ]:
# Cell 6: Compare results table
rows = [
    ('grounding-dino-swin-t',          result['status'],      result.get('num_detections',0), result.get('latency_ms','N/A'),  'Apache-2.0'),
    ('grounding-dino-original-swin-t', orig_result['status'], orig_result.get('num_detections',0), orig_result.get('latency_ms','N/A'), 'gated'),
]

print(f'{"Model":<40} {"Status":<16} {"Det":<6} {"ms":<10} {"License"}')
print('-' * 85)
for r in rows:
    print(f'{r[0]:<40} {r[1]:<16} {str(r[2]):<6} {str(r[3]):<10} {r[4]}')

In [ ]:
# Cell 7: VisionServeX dino detect usage
# High-level API
det = vsx.detect(
    model='grounding-dino-swin-t',
    image=image,
    prompt='a cat . a dog .'
)
print(f'vsx.detect status   : {det["status"]}')
print(f'vsx.detect result   : {det.get("num_detections", 0)} detections')

## Gated Models

Some GroundingDINO variants (e.g., `grounding-dino-original-swin-t`) are hosted
on gated HuggingFace repos. To access them:

1. Accept the license at `https://huggingface.co/IDEA-Research/grounding-dino-original-swin-t`
2. Set your HF token: `os.environ['HF_TOKEN'] = 'hf_...'`
3. Run: `visionservex pull grounding-dino-original-swin-t`

## Next Steps

- See `07_grounding_dino_sam_pipeline.ipynb` for the full GD+SAM pipeline.
- For DINOv2/DINOv3 embeddings, see `05_dinov3_license_aware.ipynb`.